# Gold Layer Pipeline — Olist E-Commerce Analytics

Production Gold layer for the Olist Brazilian E-Commerce dataset.
Transforms Silver tables into business-ready fact tables, dimensions, and aggregates for Databricks SQL dashboards.

**Architecture: 2 Facts + 5 Dimensions + 4 Aggregates**
- **Facts:** `fact_order_items`, `fact_orders`
- **Dimensions:** `dim_customers`, `dim_products`, `dim_sellers`, `dim_date`, `dim_geolocation`
- **Aggregates:** `agg_daily_sales`, `agg_monthly_category_revenue`, `agg_seller_performance`, `agg_customer_summary`

**Flow:** Bronze → Silver → **Gold** → Databricks SQL Dashboards

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

GOLD = "ecommerce_data.gold"
SILVER = "ecommerce_data.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")

def write_gold(df, table_name):
    """Write a DataFrame as a managed Delta table in the Gold schema."""
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(f"{GOLD}.{table_name}"))
    count = spark.table(f"{GOLD}.{table_name}").count()
    print(f"  \u2713 {table_name}: {count:,} rows -> {GOLD}.{table_name}")

In [0]:
# ── dim_date ────────────────────────────────────────────────────────────
# Calendar dimension for time-based analysis.
# Grain: one row per calendar day | Key: date
# Source: derived from orders.order_purchase_timestamp date range
# ────────────────────────────────────────────────────────────────────────
print("Building dim_date...")

bounds = spark.table(f"{SILVER}.orders").agg(
    F.min(F.to_date("order_purchase_timestamp")).alias("min_d"),
    F.max(F.to_date("order_purchase_timestamp")).alias("max_d"),
).collect()[0]

dim_date = (
    spark.sql(f"""
        SELECT explode(sequence(
            DATE '{bounds["min_d"]}', DATE '{bounds["max_d"]}', INTERVAL 1 DAY
        )) AS date
    """)
    .select(
        F.col("date"),
        F.year("date").alias("year"),
        F.month("date").alias("month"),
        F.date_format("date", "MMMM").alias("month_name"),
        F.dayofmonth("date").alias("day"),
        F.quarter("date").alias("quarter"),
        F.dayofweek("date").alias("dayofweek"),
        F.date_format("date", "EEEE").alias("dayofweek_name"),
        F.when(F.dayofweek("date").isin(1, 7), True).otherwise(False).alias("is_weekend"),
        F.date_format("date", "yyyy-MM").alias("year_month"),
    )
)
write_gold(dim_date, "dim_date")

In [0]:
# ── dim_geolocation ─────────────────────────────────────────────────────
# Geography dimension aggregated from 738K rows to one row per zip code.
# Grain: one row per zip code prefix | Key: geolocation_zip_code_prefix
# Source: geolocation (aggregated: avg lat/lng, mode city/state per zip)
# ────────────────────────────────────────────────────────────────────────
print("Building dim_geolocation...")

geo = spark.table(f"{SILVER}.geolocation")

# Average coordinates per zip (slight variation within zip areas)
geo_coords = geo.groupBy("geolocation_zip_code_prefix").agg(
    F.round(F.avg("geolocation_lat"), 6).alias("geolocation_lat"),
    F.round(F.avg("geolocation_lng"), 6).alias("geolocation_lng"),
)

# Most common city per zip via window-ranked mode
geo_city = (
    geo.groupBy("geolocation_zip_code_prefix", "geolocation_city")
    .count()
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("geolocation_zip_code_prefix").orderBy(F.desc("count"))))
    .filter("rn = 1")
    .select("geolocation_zip_code_prefix", "geolocation_city")
)

# Most common state per zip
geo_state = (
    geo.groupBy("geolocation_zip_code_prefix", "geolocation_state")
    .count()
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("geolocation_zip_code_prefix").orderBy(F.desc("count"))))
    .filter("rn = 1")
    .select("geolocation_zip_code_prefix", "geolocation_state")
)

dim_geolocation = (
    geo_coords
    .join(geo_city, "geolocation_zip_code_prefix")
    .join(geo_state, "geolocation_zip_code_prefix")
)
write_gold(dim_geolocation, "dim_geolocation")

In [0]:
# ── dim_customers ───────────────────────────────────────────────────────
# Customer dimension at the unique-customer grain.
# Silver customers is at customer_id grain (one per order, 99,441 rows).
# Gold deduplicates to customer_unique_id (96,096 real customers).
# Grain: one row per customer_unique_id | Key: customer_unique_id
# Source: customers + orders (for first/last order dates, total orders)
# ────────────────────────────────────────────────────────────────────────
print("Building dim_customers...")

cust_silver = spark.table(f"{SILVER}.customers")
orders_silver = spark.table(f"{SILVER}.orders")

# Link customers to order dates for lifecycle metrics
cust_orders = cust_silver.join(
    orders_silver.select("order_id", "customer_id", "order_purchase_timestamp"),
    "customer_id", "left",
)

# Most common city per unique customer (mode via window)
cust_city = (
    cust_silver.groupBy("customer_unique_id", "customer_city")
    .count()
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("customer_unique_id").orderBy(F.desc("count"))))
    .filter("rn = 1")
    .select("customer_unique_id", "customer_city")
)

cust_state = (
    cust_silver.groupBy("customer_unique_id", "customer_state")
    .count()
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("customer_unique_id").orderBy(F.desc("count"))))
    .filter("rn = 1")
    .select("customer_unique_id", "customer_state")
)

# Aggregate order activity per unique customer
cust_activity = cust_orders.groupBy("customer_unique_id").agg(
    F.min("order_purchase_timestamp").alias("first_order_date"),
    F.max("order_purchase_timestamp").alias("last_order_date"),
    F.countDistinct("order_id").alias("total_orders"),
)

cust_zip = (
    cust_silver.groupBy("customer_unique_id")
    .agg(F.first("customer_zip_code_prefix").alias("customer_zip_code_prefix"))
)

dim_customers = (
    cust_activity
    .join(cust_city, "customer_unique_id")
    .join(cust_state, "customer_unique_id")
    .join(cust_zip, "customer_unique_id")
    .select(
        "customer_unique_id", "customer_city", "customer_state",
        "customer_zip_code_prefix", "first_order_date", "last_order_date",
        "total_orders",
    )
)
write_gold(dim_customers, "dim_customers")

In [0]:
# ── dim_products ────────────────────────────────────────────────────────
# Product dimension with English category names and computed volume.
# Grain: one row per product_id | Key: product_id
# Source: products + category_translation (left join for English names)
# Note: 74 categories in products, 71 translations — 3 will be null.
# ────────────────────────────────────────────────────────────────────────
print("Building dim_products...")

products_silver = spark.table(f"{SILVER}.products")
cat_trans = spark.table(f"{SILVER}.category_translation")

dim_products = (
    products_silver
    .join(cat_trans, "product_category_name", "left")
    .withColumn(
        "product_volume_cm3",
        (F.col("product_length_cm") * F.col("product_height_cm") * F.col("product_width_cm"))
        .cast("int"),
    )
    .select(
        "product_id",
        "product_category_name",
        "product_category_name_english",
        "product_name_length",
        "product_description_length",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
        "product_volume_cm3",
    )
)
write_gold(dim_products, "dim_products")

In [0]:
# ── dim_sellers ─────────────────────────────────────────────────────────
# Seller dimension — clean lookup table.
# Grain: one row per seller_id | Key: seller_id
# Source: sellers (already at correct grain in Silver)
# ────────────────────────────────────────────────────────────────────────
print("Building dim_sellers...")

dim_sellers = spark.table(f"{SILVER}.sellers")
write_gold(dim_sellers, "dim_sellers")

In [0]:
# ── fact_order_items ────────────────────────────────────────────────────
# Item-level sales fact — the most granular fact table.
# Grain: one row per order item (order_id × order_item_id)
# Source: order_items ⋈ orders ⋈ customers ⋈ products ⋈ category_translation ⋈ sellers
# Join strategy: all joins are 1:1 lookups from the base (order_items) —
#   orders has unique order_id, products has unique product_id,
#   sellers has unique seller_id, customers has unique customer_id.
#   No row multiplication risk.
# Measures: price, freight_value, total_item_value
# ────────────────────────────────────────────────────────────────────────
print("Building fact_order_items...")

oi = spark.table(f"{SILVER}.order_items")
orders_silver = spark.table(f"{SILVER}.orders")
customers_silver = spark.table(f"{SILVER}.customers")
products_silver = spark.table(f"{SILVER}.products")
cat_trans = spark.table(f"{SILVER}.category_translation")
sellers_silver = spark.table(f"{SILVER}.sellers")

fact_order_items = (
    oi
    # Join orders (1:1 on order_id) — adds order context
    .join(
        orders_silver.select(
            "order_id", "customer_id", "order_status",
            F.to_date("order_purchase_timestamp").alias("order_purchase_date"),
            "order_delivered_customer_date", "order_estimated_delivery_date",
            "delivery_duration_days", "carrier_duration_days", "estimated_vs_actual_days",
        ),
        "order_id", "left",
    )
    # Join customers (1:1 on customer_id from orders) — adds customer identity
    .join(
        customers_silver.select(
            "customer_id", "customer_unique_id", "customer_city", "customer_state",
        ),
        "customer_id", "left",
    )
    # Join products (1:1 on product_id) — adds category
    .join(
        products_silver.select("product_id", "product_category_name"),
        "product_id", "left",
    )
    # Join category_translation (1:1 on product_category_name) — English names
    .join(cat_trans, "product_category_name", "left")
    # Join sellers (1:1 on seller_id) — adds seller geography
    .join(
        sellers_silver.select("seller_id", "seller_city", "seller_state"),
        "seller_id", "left",
    )
    # Derive late-delivery flag
    .withColumn(
        "is_late_delivery",
        F.when(
            F.col("order_delivered_customer_date").isNotNull()
            & (F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date")),
            True,
        ).otherwise(False),
    )
    .select(
        "order_id", "order_item_id", "product_id", "seller_id",
        "customer_unique_id", "order_status",
        F.col("price").cast("decimal(12,2)"),
        F.col("freight_value").cast("decimal(12,2)"),
        F.col("total_item_value").cast("decimal(12,2)"),
        "shipping_limit_date", "order_purchase_date",
        "product_category_name", "product_category_name_english",
        "seller_city", "seller_state",
        "customer_city", "customer_state",
        "delivery_duration_days", "carrier_duration_days",
        "estimated_vs_actual_days", "is_late_delivery",
    )
)
write_gold(fact_order_items, "fact_order_items")

In [0]:
# ── fact_orders ─────────────────────────────────────────────────────────
# Order-level fact with aggregated payments, delivery, and review metrics.
# Grain: one row per order_id (99,441 rows, same as Silver orders)
# Source: orders + customers + AGGREGATED order_items + order_payments + order_reviews
# Join strategy: each child table is pre-aggregated to order_id grain BEFORE
#   joining to orders. This prevents row multiplication from the one-to-many
#   relationships (items, payments, reviews are all 1:many from orders).
#   - order_items aggregated → sum(price), sum(freight), count(items)
#   - order_payments aggregated → sum(payment_value), primary type, max installments
#   - order_reviews deduplicated → latest review per order (243 orders have 2)
# Measures: order_revenue, order_freight, item_count, payment_total, review_score, delivery_duration_days
# ────────────────────────────────────────────────────────────────────────
print("Building fact_orders...")

orders_silver = spark.table(f"{SILVER}.orders")
customers_silver = spark.table(f"{SILVER}.customers")
oi = spark.table(f"{SILVER}.order_items")
order_payments = spark.table(f"{SILVER}.order_payments")
order_reviews = spark.table(f"{SILVER}.order_reviews")

# Pre-aggregate order_items to order grain (prevents multiplication)
items_agg = oi.groupBy("order_id").agg(
    F.sum("price").cast("decimal(12,2)").alias("order_revenue"),
    F.sum("freight_value").cast("decimal(12,2)").alias("order_freight"),
    F.sum("total_item_value").cast("decimal(12,2)").alias("order_total_value"),
    F.count("*").alias("item_count"),
    F.countDistinct("product_id").alias("distinct_products"),
    F.countDistinct("seller_id").alias("distinct_sellers"),
)

# Pre-aggregate order_payments to order grain
payments_agg = order_payments.groupBy("order_id").agg(
    F.sum("payment_value").cast("decimal(12,2)").alias("payment_total"),
    F.max("payment_installments").alias("payment_installments"),
    F.count("*").alias("payment_count"),
)

# Get primary payment type (highest payment_value per order)
pay_window = Window.partitionBy("order_id").orderBy(F.desc("payment_value"))
primary_payment = (
    order_payments
    .withColumn("rn", F.row_number().over(pay_window))
    .filter("rn = 1")
    .select("order_id", F.col("payment_type").alias("primary_payment_type"))
)

# Get latest review per order (243 orders have 2 reviews — take latest by answer timestamp)
rev_window = Window.partitionBy("order_id").orderBy(F.desc("review_answer_timestamp"))
latest_review = (
    order_reviews
    .withColumn("rn", F.row_number().over(rev_window))
    .filter("rn = 1")
    .select(
        "order_id",
        F.col("review_score").alias("review_score"),
        F.col("review_id").alias("review_id"),
        "review_response_hours",
    )
)

# Assemble fact_orders by joining all pre-aggregated tables at order grain
fact_orders = (
    orders_silver
    .join(
        customers_silver.select(
            "customer_id", "customer_unique_id", "customer_city", "customer_state",
        ),
        "customer_id", "left",
    )
    .join(items_agg, "order_id", "left")
    .join(payments_agg, "order_id", "left")
    .join(primary_payment, "order_id", "left")
    .join(latest_review, "order_id", "left")
    .withColumn(
        "is_late_delivery",
        F.when(
            F.col("order_delivered_customer_date").isNotNull()
            & (F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date")),
            True,
        ).otherwise(False),
    )
    .withColumn("order_purchase_date", F.to_date("order_purchase_timestamp"))
    .select(
        "order_id", "customer_id", "customer_unique_id",
        "order_status", "order_purchase_timestamp", "order_purchase_date",
        "order_approved_at", "order_delivered_carrier_date",
        "order_delivered_customer_date", "order_estimated_delivery_date",
        "order_revenue", "order_freight", "order_total_value",
        "item_count", "distinct_products", "distinct_sellers",
        "payment_total", "primary_payment_type", "payment_installments",
        "review_score", "review_id", "review_response_hours",
        "delivery_duration_days", "carrier_duration_days",
        "estimated_vs_actual_days", "is_late_delivery",
        "customer_city", "customer_state",
    )
)
write_gold(fact_orders, "fact_orders")

In [0]:
# ── agg_daily_sales ────────────────────────────────────────────────────
# Daily sales KPI summary — dashboard-optimized for trend lines and KPI cards.
# Grain: one row per calendar day
# Source: fact_order_items (revenue, items) + fact_orders (orders, customers, reviews, delivery)
# Join strategy: each fact table is independently aggregated to daily grain,
#   then joined on date. No cross-table row multiplication.
# ────────────────────────────────────────────────────────────────────────
print("Building agg_daily_sales...")

f_oi = spark.table(f"{GOLD}.fact_order_items")
f_ord = spark.table(f"{GOLD}.fact_orders")

# Daily item-level metrics
daily_items = f_oi.groupBy("order_purchase_date").agg(
    F.sum("total_item_value").cast("decimal(14,2)").alias("daily_revenue"),
    F.sum("freight_value").cast("decimal(14,2)").alias("daily_freight"),
    F.count("*").alias("item_count"),
    F.countDistinct("product_id").alias("unique_products"),
)

# Daily order-level metrics
daily_orders = f_ord.groupBy("order_purchase_date").agg(
    F.count("*").alias("order_count"),
    F.countDistinct("customer_unique_id").alias("unique_customers"),
    F.round(F.avg("order_revenue"), 2).alias("avg_order_value"),
    F.round(F.avg("review_score"), 2).alias("avg_review_score"),
    F.sum(F.when(F.col("is_late_delivery") == True, 1).otherwise(0)).alias("late_delivery_count"),
    F.count(F.when(F.col("order_status") == "delivered", True)).alias("delivered_count"),
)

agg_daily_sales = (
    daily_items.join(daily_orders, "order_purchase_date", "outer")
    .withColumn(
        "late_delivery_rate",
        F.when(F.col("delivered_count") > 0,
               F.round(F.col("late_delivery_count") / F.col("delivered_count"), 4))
         .otherwise(0.0),
    )
    .select(
        F.col("order_purchase_date").alias("date"),
        "daily_revenue", "daily_freight",
        "order_count", "item_count", "unique_customers", "unique_products",
        "avg_order_value", "avg_review_score",
        "late_delivery_count",
        F.col("late_delivery_rate").cast("decimal(6,4)"),
    )
    .orderBy("date")
)
write_gold(agg_daily_sales, "agg_daily_sales")

In [0]:
# ── agg_monthly_category_revenue ────────────────────────────────────────
# Monthly revenue by product category — for category trend dashboards.
# Grain: one row per year-month × product category
# Source: fact_order_items (no joins needed — categories already denormalized)
# ────────────────────────────────────────────────────────────────────────
print("Building agg_monthly_category_revenue...")

f_oi = spark.table(f"{GOLD}.fact_order_items")

agg_monthly_category_revenue = (
    f_oi
    .withColumn("year_month", F.date_format("order_purchase_date", "yyyy-MM"))
    .groupBy("year_month", "product_category_name_english")
    .agg(
        F.sum("total_item_value").cast("decimal(14,2)").alias("monthly_revenue"),
        F.sum("price").cast("decimal(14,2)").alias("monthly_price"),
        F.sum("freight_value").cast("decimal(14,2)").alias("monthly_freight"),
        F.count("*").alias("item_count"),
        F.countDistinct("order_id").alias("order_count"),
        F.round(F.avg("price"), 2).alias("avg_item_price"),
        F.round(F.avg("freight_value"), 2).alias("avg_freight"),
    )
    .orderBy("year_month", F.desc("monthly_revenue"))
)
write_gold(agg_monthly_category_revenue, "agg_monthly_category_revenue")

In [0]:
# ── agg_seller_performance ─────────────────────────────────────────────
# Seller scorecard — revenue, volume, delivery, and review metrics per seller.
# Grain: one row per seller_id
# Source: fact_order_items (item metrics) + fact_orders (delivery & review metrics)
# Join strategy: fact_order_items is aggregated to seller grain independently.
#   For delivery/review, we take distinct (seller_id, order_id) pairs from
#   fact_order_items, join to fact_orders at order grain, then aggregate to seller.
#   No row multiplication because we join at order_id grain (1:1 from orders).
# ────────────────────────────────────────────────────────────────────────
print("Building agg_seller_performance...")

f_oi = spark.table(f"{GOLD}.fact_order_items")
f_ord = spark.table(f"{GOLD}.fact_orders")

# Item-level revenue metrics per seller
seller_items = f_oi.groupBy("seller_id").agg(
    F.sum("total_item_value").cast("decimal(14,2)").alias("seller_revenue"),
    F.sum("price").cast("decimal(14,2)").alias("seller_product_revenue"),
    F.sum("freight_value").cast("decimal(14,2)").alias("seller_freight"),
    F.count("*").alias("items_sold"),
    F.countDistinct("order_id").alias("orders_fulfilled"),
    F.countDistinct("product_id").alias("unique_products"),
    F.countDistinct("customer_unique_id").alias("unique_customers"),
    F.round(F.avg("price"), 2).alias("avg_item_price"),
)

# Order-level delivery & review metrics per seller
# Use distinct order_ids per seller to avoid item-level inflation
seller_orders = (
    f_oi.select("seller_id", "order_id").distinct()
    .join(
        f_ord.select("order_id", "is_late_delivery", "delivery_duration_days", "review_score"),
        "order_id", "left",
    )
)

seller_delivery = seller_orders.groupBy("seller_id").agg(
    F.sum(F.when(F.col("is_late_delivery") == True, 1).otherwise(0)).alias("late_deliveries"),
    F.count(F.when(F.col("delivery_duration_days").isNotNull(), True)).alias("delivered_orders"),
    F.round(F.avg("delivery_duration_days"), 1).alias("avg_delivery_days"),
    F.round(F.avg("review_score"), 2).alias("avg_review_score"),
)

agg_seller_performance = (
    seller_items
    .join(seller_delivery, "seller_id", "left")
    .withColumn(
        "late_delivery_rate",
        F.when(F.col("delivered_orders") > 0,
               F.round(F.col("late_deliveries") / F.col("delivered_orders"), 4))
         .otherwise(0.0),
    )
    .withColumn(
        "avg_order_value",
        F.when(F.col("orders_fulfilled") > 0,
               F.round(F.col("seller_revenue") / F.col("orders_fulfilled"), 2))
         .otherwise(0.0),
    )
    .select(
        "seller_id",
        "seller_revenue", "seller_product_revenue", "seller_freight",
        "items_sold", "orders_fulfilled", "unique_products", "unique_customers",
        "avg_item_price", "avg_order_value",
        "avg_review_score", "avg_delivery_days",
        "late_deliveries", "delivered_orders",
        F.col("late_delivery_rate").cast("decimal(6,4)"),
    )
    .orderBy(F.desc("seller_revenue"))
)
write_gold(agg_seller_performance, "agg_seller_performance")

In [0]:
# ── agg_customer_summary ───────────────────────────────────────────────
# Customer CLV and segmentation — for customer analytics dashboards.
# Grain: one row per customer_unique_id
# Source: fact_orders (all metrics already at order grain)
# Join strategy: fact_orders is aggregated to customer_unique_id grain.
#   No joins needed — fact_orders already has customer_unique_id.
# Segments: VIP (3+ orders & 500+ revenue), Returning (2+ orders), New (1 order)
# ────────────────────────────────────────────────────────────────────────
print("Building agg_customer_summary...")

f_ord = spark.table(f"{GOLD}.fact_orders")

agg_customer_summary = (
    f_ord.groupBy("customer_unique_id").agg(
        F.sum("order_revenue").cast("decimal(14,2)").alias("customer_revenue"),
        F.count("*").alias("order_count"),
        F.round(F.avg("order_revenue"), 2).alias("avg_order_value"),
        F.min("order_purchase_date").alias("first_order_date"),
        F.max("order_purchase_date").alias("last_order_date"),
        F.round(F.avg("review_score"), 2).alias("avg_review_score"),
        F.sum("item_count").alias("total_items"),
        F.first("customer_city").alias("customer_city"),
        F.first("customer_state").alias("customer_state"),
    )
    .withColumn("is_repeat_customer", F.when(F.col("order_count") > 1, True).otherwise(False))
    .withColumn("days_active", F.datediff("last_order_date", "first_order_date"))
    .withColumn(
        "customer_segment",
        F.when(
            (F.col("order_count") >= 3) & (F.col("customer_revenue") >= 500), "VIP"
        ).when(
            F.col("order_count") > 1, "Returning"
        ).otherwise("New"),
    )
    .orderBy(F.desc("customer_revenue"))
)
write_gold(agg_customer_summary, "agg_customer_summary")

In [0]:
# ── VALIDATION SUITE ────────────────────────────────────────────────────
# Tests: row counts, duplicate keys, null keys, referential integrity,
# revenue reconciliation, order-count reconciliation, double-counting detection
# ────────────────────────────────────────────────────────────────────────
print("=" * 80)
print("GOLD LAYER VALIDATION")
print("=" * 80)

# 1. Row count summary
print("\n1. ROW COUNTS")
print("-" * 40)
gold_tables = [
    "dim_date", "dim_geolocation", "dim_customers", "dim_products", "dim_sellers",
    "fact_order_items", "fact_orders",
    "agg_daily_sales", "agg_monthly_category_revenue",
    "agg_seller_performance", "agg_customer_summary",
]
for t in gold_tables:
    cnt = spark.table(f"{GOLD}.{t}").count()
    print(f"  {t}: {cnt:,} rows")

# 2. Duplicate primary key checks
print("\n2. DUPLICATE KEY CHECKS")
print("-" * 40)
dup_checks = [
    ("dim_date", "date"),
    ("dim_geolocation", "geolocation_zip_code_prefix"),
    ("dim_customers", "customer_unique_id"),
    ("dim_products", "product_id"),
    ("dim_sellers", "seller_id"),
    ("fact_orders", "order_id"),
    ("agg_customer_summary", "customer_unique_id"),
    ("agg_seller_performance", "seller_id"),
]
for table, key in dup_checks:
    total = spark.table(f"{GOLD}.{table}").count()
    distinct = spark.table(f"{GOLD}.{table}").select(key).distinct().count()
    status = "OK" if total == distinct else f"FAIL ({total - distinct} dups)"
    print(f"  {table}.{key}: {status}")

# Composite key check for fact_order_items
print(f"  fact_order_items (order_id, order_item_id): ", end="")
f_oi = spark.table(f"{GOLD}.fact_order_items")
total_oi = f_oi.count()
distinct_oi = f_oi.select("order_id", "order_item_id").distinct().count()
print("OK" if total_oi == distinct_oi else f"FAIL ({total_oi - distinct_oi} dups)")

# 3. Null key checks
print("\n3. NULL KEY CHECKS")
print("-" * 40)
null_checks = [
    ("fact_order_items", ["order_id", "order_item_id", "product_id", "seller_id"]),
    ("fact_orders", ["order_id", "customer_unique_id"]),
    ("dim_products", ["product_id"]),
    ("dim_customers", ["customer_unique_id"]),
]
for table, keys in null_checks:
    df = spark.table(f"{GOLD}.{table}")
    for k in keys:
        nulls = df.filter(F.col(k).isNull()).count()
        print(f"  {table}.{k}: {nulls} nulls -> {'OK' if nulls == 0 else 'FAIL'}")

# 4. Revenue reconciliation (Gold vs Silver)
print("\n4. REVENUE RECONCILIATION")
print("-" * 40)
silver_rev = spark.table(f"{SILVER}.order_items").agg(F.sum("total_item_value")).collect()[0][0]
gold_rev = f_oi.agg(F.sum("total_item_value")).collect()[0][0]
diff = abs(float(silver_rev) - float(gold_rev))
print(f"  Silver order_items total_item_value: {float(silver_rev):,.2f}")
print(f"  Gold fact_order_items total_item_value: {float(gold_rev):,.2f}")
print(f"  Difference: {diff:,.2f} -> {'OK' if diff < 0.01 else 'FAIL'}")

# 5. Order count reconciliation
print("\n5. ORDER COUNT RECONCILIATION")
print("-" * 40)
silver_orders = spark.table(f"{SILVER}.orders").count()
gold_orders = spark.table(f"{GOLD}.fact_orders").count()
print(f"  Silver orders: {silver_orders:,}")
print(f"  Gold fact_orders: {gold_orders:,}")
print(f"  Match: {'OK' if silver_orders == gold_orders else 'FAIL'}")

# 6. Double-counting detection (item_count in fact_orders vs Silver order_items)
print("\n6. DOUBLE-COUNTING DETECTION")
print("-" * 40)
silver_items_per_order = (
    spark.table(f"{SILVER}.order_items")
    .groupBy("order_id").agg(F.count("*").alias("silver_items"))
)
gold_items_per_order = spark.table(f"{GOLD}.fact_orders").select("order_id", "item_count")
compare = silver_items_per_order.join(gold_items_per_order, "order_id", "inner")
mismatches = compare.filter(F.col("silver_items") != F.col("item_count")).count()
total_compared = compare.count()
print(f"  Orders compared: {total_compared:,}")
print(f"  Item count mismatches: {mismatches}")
print(f"  Result: {'OK — no double counting' if mismatches == 0 else 'FAIL — potential double counting'}")

# 7. Referential integrity
print("\n7. REFERENTIAL INTEGRITY")
print("-" * 40)
f_ord = spark.table(f"{GOLD}.fact_orders")
orphan_orders = f_oi.join(f_ord, "order_id", "left_anti").count()
print(f"  fact_order_items -> fact_orders (orphaned order_id): {orphan_orders} -> {'OK' if orphan_orders == 0 else 'FAIL'}")

orphan_products = f_oi.join(spark.table(f"{GOLD}.dim_products"), "product_id", "left_anti").count()
print(f"  fact_order_items -> dim_products (orphaned product_id): {orphan_products} -> {'OK' if orphan_products == 0 else 'FAIL'}")

orphan_sellers = f_oi.join(spark.table(f"{GOLD}.dim_sellers"), "seller_id", "left_anti").count()
print(f"  fact_order_items -> dim_sellers (orphaned seller_id): {orphan_sellers} -> {'OK' if orphan_sellers == 0 else 'FAIL'}")

orphan_customers = f_ord.join(spark.table(f"{GOLD}.dim_customers"), "customer_unique_id", "left_anti").count()
print(f"  fact_orders -> dim_customers (orphaned customer_unique_id): {orphan_customers} -> {'OK' if orphan_customers == 0 else 'FAIL'}")

print("\n" + "=" * 80)
print("VALIDATION COMPLETE")
print("=" * 80)

In [0]:
# ─────────────────────────────────────────────────────────────────────────
# LIQUID CLUSTERING — Optimize fact table query performance
# ─────────────────────────────────────────────────────────────────────────
# Applies liquid clustering to the two fact tables on their most
# commonly filtered columns. Liquid clustering continuously reorganizes
# data files to skip irrelevant rows during filtered queries.
#
# fact_order_items: clustered on (order_purchase_date, product_id)
#   — dashboard filters by date range and product_id
# fact_orders: clustered on (order_purchase_date)
#   — dashboard filters by date range
#
# Also runs OPTIMIZE to compact and cluster existing data immediately.
# ─────────────────────────────────────────────────────────────────────────

print("Applying liquid clustering to Gold fact tables...")
print()

# 1. fact_order_items — cluster on date + product_id (most common filter combo)
print("  ALTER TABLE ecommerce_data.gold.fact_order_items CLUSTER BY (order_purchase_date, product_id)...")
spark.sql("ALTER TABLE ecommerce_data.gold.fact_order_items CLUSTER BY (order_purchase_date, product_id)")
print("  ✓ Liquid clustering enabled for fact_order_items")
print()

# 2. fact_orders — cluster on date (primary dashboard filter)
print("  ALTER TABLE ecommerce_data.gold.fact_orders CLUSTER BY (order_purchase_date)...")
spark.sql("ALTER TABLE ecommerce_data.gold.fact_orders CLUSTER BY (order_purchase_date)")
print("  ✓ Liquid clustering enabled for fact_orders")
print()

# 3. Run OPTIMIZE to apply clustering to existing data
print("  OPTIMIZE ecommerce_data.gold.fact_order_items...")
spark.sql("OPTIMIZE ecommerce_data.gold.fact_order_items")
print("  ✓ fact_order_items optimized and clustered")
print()

print("  OPTIMIZE ecommerce_data.gold.fact_orders...")
spark.sql("OPTIMIZE ecommerce_data.gold.fact_orders")
print("  ✓ fact_orders optimized and clustered")
print()

print("=" * 60)
print("LIQUID CLUSTERING COMPLETE")
print("=" * 60)
print("  fact_order_items: CLUSTER BY (order_purchase_date, product_id)")
print("  fact_orders:      CLUSTER BY (order_purchase_date)")
print("  Both tables optimized with existing data compacted.")